In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Device: {DEVICE}")

# 1) Scaled Dot-Product Attention

## 1.1 Self Attention

<img src="https://pbs.twimg.com/media/FzjhZk5X0AYAs_-?format=jpg&name=4096x4096" width="600">

In [ ]:
class Attention(nn.Module):
    def __init__(self, ori_dim: int = 512, embed_dim: int = 512) -> None:
        super().__init__()
        self.embed_dim = embed_dim
        self.dim_K = torch.tensor(embed_dim)

        # Linear layers to project input features to query, key, and value spaces
        # Input: (batch_num, seq_len, model_dim)
        # Output: (batch_num, seq_len, embed_dim)
        self.query = nn.Linear(in_features=ori_dim, out_features=embed_dim, bias=True)
        self.key = nn.Linear(in_features=ori_dim, out_features=embed_dim, bias=True)
        self.value = nn.Linear(in_features=ori_dim, out_features=embed_dim, bias=True)

    def self_attention(self, Q: Tensor, K: Tensor, V: Tensor) -> Tensor:
        '''
        Compute scaled dot-product attention scores and return weighted context vectors
        '''

        # Input Q, K, V: (batch_num, seq_len, embed_dim)
        
        # Transpose key for matrix multiplication
        # (batch_num, seq_len, embed_dim) → (batch_num, embed_dim, seq_len)
        K_T = torch.transpose(K, 1, 2)

        # Compute scaled similarity scores
        # Input: (batch_num, seq_len, embed_dim) x (batch_num, embed_dim, seq_len)
        # Output: (batch_num, seq_len, seq_len)
        score = torch.matmul(Q, K_T) / torch.sqrt(self.dim_K)

        # Apply softmax to normalize attention weights across key positions
        # (batch_num, seq_len, seq_len)
        score = torch.softmax(score, dim=-1)

        # Compute weighted sum of values
        # Input: (batch_num, seq_len, seq_len) x (batch_num, seq_len, embed_dim)
        # Output: (batch_num, seq_len, embed_dim)
        Z = torch.matmul(score, V)
        return Z

    def forward(self, x: Tensor) -> Tensor:
        # Project input to Q, K, V then apply self-attention
        # Input: (batch_num, seq_len, model_dim)

        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, embed_dim)
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        # Apply scaled dot-product attention
        # Q/K/V=(batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        Z = self.self_attention(Q, K, V)
        return Z

## 1.2 Multi-head Attention

In [ ]:
class MultiheadAttention(nn.Module):
    """https://arxiv.org/abs/1706.03762"""

    def __init__(self, embed_dim: int = 512, n_head: int = 8) -> None:
        super().__init__()
        assert embed_dim % n_head == 0, "embed_dim must be divisible by n_head"
        self.n_head = n_head
        self.embed_dim = embed_dim

        # Input: (batch_num, seq_len, embed_dim)
        # Output: (batch_num, seq_len, embed_dim // num_heads)
        self.multihead = nn.ModuleList(
            [Attention(embed_dim, embed_dim // n_head) for _ in range(n_head)]
        )

        # Output projection W^O, mixes information across heads
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x: Tensor) -> Tensor:
        # Process input through num_heads attention heads and concatenate along feature dim
        # Input: (batch_num, seq_len, embed_dim)

        # Concatenate all head outputs along the last dimension
        # Input: num_heads * (batch_num, seq_len, embed_dim // num_heads)
        # Output: (batch_num, seq_len, embed_dim)
        Z = torch.cat([head(x) for head in self.multihead], dim=-1)

        # (batch_num, seq_len, embed_dim)
        Z = self.out_proj(Z)

        return Z

In [ ]:
class MultiQueryAttention(nn.Module):
    """Multi-Query Attention (shared K/V across heads)"""

    def __init__(self, embed_dim: int = 512, n_head: int = 8) -> None:
        super().__init__()
        assert embed_dim % n_head == 0, "embed_dim must be divisible by n_head"

        self.n_head = n_head
        self.embed_dim = embed_dim
        self.head_dim = embed_dim // n_head

        # Q projection: separate query for each head
        # (batch_num, seq_len, embed_dim)
        self.q_proj = nn.Linear(embed_dim, embed_dim)

        # Shared K and V projections (single KV head, broadcast to all query heads)
        # Input: (batch_num, seq_len, embed_dim)
        # Output: (batch_num, seq_len, embed_dim // num_heads)
        self.k_proj = nn.Linear(embed_dim, self.head_dim)
        self.v_proj = nn.Linear(embed_dim, self.head_dim)

        # Final output projection
        # (batch_num, seq_len, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x: Tensor) -> Tensor:
        # (batch_num, seq_len, embed_dim)
        batch_size, seq_len, _ = x.shape

        # Project and reshape query heads
        # Input: (batch_num, seq_len, embed_dim)
        # Output: (batch_num, num_heads, seq_len, embed_dim // num_heads)
        Q = (
            self.q_proj(x)
            .view(batch_size, seq_len, self.n_head, self.head_dim)
            .transpose(1, 2)
        )

        # Shared K and V: project to single head then broadcast across all query heads
        # Input: (batch_num, seq_len, embed_dim)
        # Output: (batch_num, 1, seq_len, embed_dim // num_heads)
        K = self.k_proj(x).unsqueeze(1)
        V = self.v_proj(x).unsqueeze(1)

        # Compute scaled dot-product attention scores
        # Q=(batch_num, num_heads, seq_len, embed_dim // num_heads)
        # K=(batch_num, 1, seq_len, embed_dim // num_heads)
        # Output: (batch_num, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # (batch_num, num_heads, seq_len, seq_len)
        attn = F.softmax(scores, dim=-1)

        # Compute weighted sum of shared values
        # Input: attn=(batch_num, num_heads, seq_len, seq_len)
        # Input: V=(batch_num, 1, seq_len, embed_dim // num_heads)
        # Output: (batch_num, num_heads, seq_len, embed_dim // num_heads)
        Z = torch.matmul(attn, V)

        # Merge all heads back into full embedding dimension
        # Input: (batch_num, num_heads, seq_len, embed_dim // num_heads)
        # Output: (batch_num, seq_len, embed_dim)
        Z = Z.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim)

        # Final linear projection
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        Z = self.out_proj(Z)
        return Z

In [6]:
class GroupedQueryAttention(nn.Module):
    """Grouped-Query Attention (shared K/V within groups)"""

    def __init__(
        self, embed_dim: int = 512, n_head: int = 8, n_kv_head: int = 4
    ) -> None:
        super().__init__()
        assert embed_dim % n_head == 0, "embed_dim must be divisible by n_head"
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"

        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.embed_dim = embed_dim
        self.head_dim = embed_dim // n_head
        self.n_query_each_group = n_head // n_kv_head

        # Q projection: one query vector per query head
        # Input: (batch_num, seq_len, embed_dim) → Output: (batch_num, seq_len, embed_dim)
        self.q_proj = nn.Linear(embed_dim, embed_dim)

        # Group-shared K and V projections (one K/V vector per KV-head group)
        # Input: (batch_num, seq_len, embed_dim)
        # Output: (batch_num, seq_len, n_kv_head * embed_dim // num_heads)
        self.k_proj = nn.Linear(embed_dim, n_kv_head * self.head_dim)
        self.v_proj = nn.Linear(embed_dim, n_kv_head * self.head_dim)

        # Final output projection
        # Input: (batch_num, seq_len, embed_dim) → Output: (batch_num, seq_len, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x: Tensor) -> Tensor:
        # Input: (batch_num, seq_len, embed_dim)
        # Output: (batch_num, seq_len, embed_dim)
        batch_size, seq_len, _ = x.shape

        # Project and reshape query heads
        # Input: (batch_num, seq_len, embed_dim)
        # Output: (batch_num, num_heads, seq_len, embed_dim // num_heads)
        Q = (
            self.q_proj(x)
            .view(batch_size, seq_len, self.n_head, self.head_dim)
            .transpose(1, 2)
        )

        # Project and reshape grouped K and V heads
        # Input: (batch_num, seq_len, embed_dim)
        # Output: (batch_num, n_kv_head, seq_len, embed_dim // num_heads)
        K = (
            self.k_proj(x)
            .view(batch_size, seq_len, self.n_kv_head, self.head_dim)
            .transpose(1, 2)
        )
        V = (
            self.v_proj(x)
            .view(batch_size, seq_len, self.n_kv_head, self.head_dim)
            .transpose(1, 2)
        )

        # Expand grouped K/V so each query head shares the K/V from its group
        # Input: (batch_num, n_kv_head, seq_len, embed_dim // num_heads)
        # Output: (batch_num, num_heads, seq_len, embed_dim // num_heads)
        K = K.repeat_interleave(self.n_query_each_group, dim=1)
        V = V.repeat_interleave(self.n_query_each_group, dim=1)

        # Compute scaled dot-product attention scores
        # Q=(batch_num, num_heads, seq_len, embed_dim // num_heads)
        # K=(batch_num, num_heads, seq_len, embed_dim // num_heads)
        # Output: (batch_num, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        # Output: (batch_num, num_heads, seq_len, seq_len)
        attn = F.softmax(scores, dim=-1)

        # Compute weighted sum of values
        # attn=(batch_num, num_heads, seq_len, seq_len)
        # V=(batch_num, num_heads, seq_len, embed_dim // num_heads)
        # Output: (batch_num, num_heads, seq_len, embed_dim // num_heads)
        Z = torch.matmul(attn, V)

        # Merge all heads back into full embedding dimension
        # Input: (batch_num, num_heads, seq_len, embed_dim // num_heads)
        # Output: (batch_num, seq_len, embed_dim)
        Z = Z.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim)

        # Final linear projection
        # Input: (batch_num, seq_len, embed_dim) → Output: (batch_num, seq_len, embed_dim)
        Z = self.out_proj(Z)
        return Z

# 2) Attention Complexity and Efficiency Bottlenecks

- The standard scaled dot-product attention computes pairwise scores between all tokens in a sequence.
- This results in a time and memory complexity of $O(n^2d)$, where $n$ is the sequence length and $d$ is the head dimension.
- For long sequences (e.g., in long-context LLMs), the $n^2$ term dominates, leading to massive memory consumption.
- Furthermore, the true bottleneck on modern hardware (like GPUs) is often not compute, but memory bandwidth — moving the $n \times n$ attention matrix between High Bandwidth Memory (HBM) and the fast compute SRAM.

**Sample Input Output**
- Input: Sequence length $n=8192$, Head dimension $d=128$.
- Output: Attention matrix size $8192 \times 8192$ floats $\approx 268$ MB per head per layer.
- For a batch of 32 and 32 heads, this is over 274 GB of memory just to store intermediate attention scores for one layer.


To scale transformers to longer sequences, we must avoid materializing the full $O(n^2)$ attention matrix in HBM.

1. Techniques like Flash Attention restructure the computation to be IO-aware
2. While sparse attention patterns (like sliding windows) reduce the theoretical complexity from $O(n^2)$ to $O(n \times w)$, where $w$ is the window size.

## 2.1 Flash Attention

- Standard attention materializes the intermediate $Q K^T$ matrix to HBM to compute the softmax, then loads it back to multiply with $V$. This causes a memory bandwidth bottleneck.
- Flash Attention (Dao et al., 2022: *FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness*) avoids this by computing attention in blocks (tiling). It loads blocks of $Q$, $K$, and $V$ from HBM to SRAM, computes a running softmax, multiplies with $V$ in SRAM, and writes the final output back to HBM.

**Sample Input Output**
- Input: $Q$, $K$, $V$ tensors of shape `(batch_num, seq_len, embed_dim)`.
- Output: $Z$ tensor of shape `(batch_num, seq_len, embed_dim)`, mathematically equivalent to standard attention but computed block-wise.

- Flash Attention requires maintaining a running maximum and a running exponential sum to compute the softmax incrementally over tiles.
- The implementation below is a purely Python-based pedagogical simulation of the block-wise forward pass.
- While not actually faster without a custom CUDA kernel, it demonstrates the exact mathematics of tiling and the running softmax mechanism.

In [7]:
def simulated_flash_attention_forward(
    Q: Tensor, K: Tensor, V: Tensor, block_size: int = 32
):
    batch_num, seq_len, embed_dim = Q.shape

    # Initialize output tensor in HBM
    # (batch_num, seq_len, embed_dim)
    O = torch.zeros_like(Q)

    # Initialize running max and sum for softmax denominator
    # (batch_num, seq_len) -> (batch_num, seq_len, 1)
    m = torch.full((batch_num, seq_len, 1), float("-inf"), device=Q.device)
    l = torch.zeros((batch_num, seq_len, 1), device=Q.device)

    # Split tensors into blocks
    num_blocks = (seq_len + block_size - 1) // block_size

    for j in range(num_blocks):
        start_j = j * block_size
        end_j = min(start_j + block_size, seq_len)

        # Load block of K and V to SRAM
        # (batch_num, block_size, embed_dim)
        K_j = K[:, start_j:end_j, :]
        V_j = V[:, start_j:end_j, :]

        for i in range(num_blocks):
            start_i = i * block_size
            end_i = min(start_i + block_size, seq_len)

            # Load block of Q to SRAM
            # Input: (batch_num, seq_len, embed_dim)
            # Output: (batch_num, block_size, embed_dim)
            Q_i = Q[:, start_i:end_i, :]

            # Compute block dot product
            # Input: (batch_num, block_size, embed_dim) x (batch_num, embed_dim, block_size)
            # Output: (batch_num, block_size, block_size)
            S_ij = torch.bmm(Q_i, K_j.transpose(1, 2)) / math.sqrt(embed_dim)

            # Load previous running statistics
            # (batch_num, seq_len, 1) -> (batch_num, block_size, 1)
            m_i = m[:, start_i:end_i, :]
            l_i = l[:, start_i:end_i, :]

            # Compute new running max
            # Input: (batch_num, block_size, block_size)
            # Output: (batch_num, block_size, 1)
            m_ij = torch.max(S_ij, dim=-1, keepdim=True).values
            m_new = torch.maximum(m_i, m_ij)

            # Compute exponentiated scores stabilized by the new max
            # Input: (batch_num, block_size, block_size)
            # Output: (batch_num, block_size, block_size)
            P_ij = torch.exp(S_ij - m_new)

            # Update running sum
            # (batch_num, block_size, 1)
            l_new = torch.exp(m_i - m_new) * l_i + torch.sum(P_ij, dim=-1, keepdim=True)

            # Load previous output block
            # Input: (batch_num, seq_len, embed_dim)
            # Output: (batch_num, block_size, embed_dim)
            O_i = O[:, start_i:end_i, :]

            # Update output block
            # Input: (batch_num, block_size, embed_dim)
            # Output: (batch_num, block_size, embed_dim)
            O_new = torch.exp(m_i - m_new) * O_i + torch.bmm(P_ij, V_j)

            # Write updated blocks back to HBM
            O[:, start_i:end_i, :] = O_new
            m[:, start_i:end_i, :] = m_new
            l[:, start_i:end_i, :] = l_new

    # Final normalization step
    # Input (batch_num, seq_len, embed_dim) / (batch_num, seq_len, 1)
    # Output: (batch_num, seq_len, embed_dim)
    O = O / l

    return O


# Test the simulated Flash Attention
Q = torch.randn(2, 64, 128)
K = torch.randn(2, 64, 128)
V = torch.randn(2, 64, 128)

out_flash = simulated_flash_attention_forward(Q, K, V, block_size=16)

# Verify correctness against standard attention
S = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(128)
P = torch.softmax(S, dim=-1)
out_std = torch.bmm(P, V)

print(
    "Max difference between standard and simulated Flash Attention:",
    torch.max(torch.abs(out_flash - out_std)).item(),
)

Max difference between standard and simulated Flash Attention: 2.980232238769531e-07


## 2.2 Sliding Window and Sparse Attention

1. Not all tasks require global attention. In many sequences, local context is sufficient for token representations.
2. Sliding window attention restricts the receptive field of each query to a fixed neighborhood of surrounding keys, reducing complexity from $O(n^2)$ to $O(n \times w)$ where $w$ is the window size.
3. This was popularized by models like Longformer (Beltagy et al., 2020) and Sparse Transformers (Child et al., 2019).
4. We create a banded matrix mask based on the distance between query index $i$ and key index $j$. If $|i - j| > \text{window\_size} // 2$, we apply a large negative value to the attention score before softmax, effectively pruning those connections.
5. While this dense implementation still creates an $n \times n$ matrix in PyTorch (for educational clarity), optimized kernels (like BlockSparse) only compute the non-zero blocks.

**Sample Input Output**
- Input: Sequence length $n=10$, Window size $w=3$ (1 token left, 1 token right, plus itself).
- Output: An attention mask where only a narrow diagonal band is True (allowed), and the rest is False (-inf mask).

In [ ]:
class SlidingWindowAttention(nn.Module):
    def __init__(self, embed_dim: int, window_size: int):
        super().__init__()
        self.embed_dim = embed_dim
        self.window_size = window_size
        self.scale = math.sqrt(embed_dim)

        # Linear projections for query, key, value
        # (batch_num, seq_len, embed_dim)
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x: Tensor) -> Tensor:
        batch_num, seq_len, embed_dim = x.shape

        # Project inputs
        # (batch_num, seq_len, embed_dim)
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        # Compute dense scaled dot-product scores
        # Input: (batch_num, seq_len, embed_dim) x (batch_num, embed_dim, seq_len)
        # Output: (batch_num, seq_len, seq_len)
        scores = torch.bmm(Q, K.transpose(1, 2)) / self.scale

        # Generate sliding window mask
        # (seq_len, 1) - (1, seq_len) -> (seq_len, seq_len)
        idx = torch.arange(seq_len, device=x.device)
        distance = torch.abs(idx.unsqueeze(1) - idx.unsqueeze(0))

        # Create boolean mask where True means to drop the score
        # (seq_len, seq_len) -> (seq_len, seq_len)
        mask = distance > (self.window_size // 2)

        # Apply mask by setting outside-window elements to -infinity
        # (batch_num, seq_len, seq_len) -> (batch_num, seq_len, seq_len)
        scores = scores.masked_fill(mask.unsqueeze(0), float("-inf"))

        # Compute attention probabilities
        # (batch_num, seq_len, seq_len) -> (batch_num, seq_len, seq_len)
        attn_weights = F.softmax(scores, dim=-1)

        # Multiply by values to get context
        # Input (batch_num, seq_len, seq_len) x (batch_num, seq_len, embed_dim)
        # Output: (batch_num, seq_len, embed_dim)
        output = torch.bmm(attn_weights, V)

        return output


# Demonstrate Sliding Window Attention
x = torch.randn(1, 10, 64)
window_attn = SlidingWindowAttention(embed_dim=64, window_size=3)
out_sparse = window_attn(x)

print("Output shape of Sliding Window Attention:", out_sparse.shape)

Output shape of Sliding Window Attention: torch.Size([1, 10, 64])
